# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [36]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [37]:
import sys
sys.path.append('../05_src/')

In [38]:
from openai import OpenAI
from pydantic import BaseModel, Field
from utils.clients import get_client
import os
# os.environ["LANGSMITH_TRACING"] = "false"
print(os.getcwd())
from utils.logger import get_logger
from utils.clients import get_client
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
_logs = get_logger(__name__)
client = get_client()

/Users/ucladev/UofT-DSI/repos/deploying-ai/02_activities


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [39]:
import pypdf
from langchain_core.documents import Document

def load_pdf_pages(file_path: str) -> list[Document]:
    reader = pypdf.PdfReader(file_path)
    return [
        Document(
            page_content=page.extract_text() or "",
            metadata={"source": file_path, "page": i},
        )
        for i, page in enumerate(reader.pages)
    ]

file_path = "./documents/managing_oneself.pdf"
docs = load_pdf_pages(file_path)

In [40]:
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [41]:
class article(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    token_limit: int
    tone: str

In [86]:
token_limit = 1000
response = client.chat.completions.parse(
    model=MODEL,

        messages=[
                {
                    "role": "system",
                    "content": f"""Act as a publication reviewer and journalist.
                    Use STRICTLY Bureaucratese to write a 'summary' of Managing Oneself ensuring keypoints are captured.
                    Very important: Capture all the key points that are available as extra information as well as the main topics on the article so that someone not familiar with the topic, can master it.
                    Address key questions about the representation of Ducker's ideas and the specific section 'The Second Half of Your Life', indicating a lack of comprehensive coverage of the original content.
                    Return the information in accordance with the structured format not exceeding {token_limit}."""
                },
                {
                    "role": "user",
                    "content": f"""Return the 'author', 'title', 'relevance', 'summary', and 'tone'.

        Document:
        {document_text}"""
                },
            ],
    response_format=article,
    max_tokens=token_limit,
)

event = response.choices[0].message.parsed

In [87]:
print(event.summary)
print(f'Author: {event.author}')
print(f'Title: {event.title}')
print(f'Relevance: {event.relevance}')
print(f'Tone: {event.tone}')

The article emphasizes the necessity for individuals, particularly knowledge workers, to take control of their careers by understanding their strengths, values, and preferred working styles. It discusses the imperative of becoming one's own chief executive officer in a professional environment where companies do not guide careers. The text highlights methods for self-assessment such as feedback analysis and poses essential reflective questions regarding strengths, performance styles, values, workplace fit, and potential contributions. Furthermore, it addresses the significance of these elements in paving the route towards both personal and professional excellence. It notably touches on 'The Second Half of Your Life,' stressing the importance of preparing for continued professional engagement beyond traditional retirement ages by undertaking new careers, parallel roles, or social entrepreneurship, thereby emphasizing that self-management is a lifelong process requiring proactive plannin

In [88]:
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
total_tokens = response.usage.total_tokens
print(f"Input: {input_tokens}, Output: {output_tokens}, Total: {total_tokens}")

Input: 12394, Output: 217, Total: 12611


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [89]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
from deepeval.models.base_model import DeepEvalBaseLLM

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

test_case = LLMTestCase(input=document_text, actual_output=event.summary)


metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary have a good representation of ideas dicussed by Ducker?",
        "Are each of Ducker's main points proportionally summarized?",
        "Does the summary discuss The Second Half of Your Life section?",
        "Is the summery written professionally?",
        "Is the summary informative in general with someone unfamiliar with the concept?",
    ],
)

# 5. Execute evaluation
# evaluate(test_cases=[test_case], metrics=[metric])

metric.measure(test_case)

print("Score:", metric.score)
print("Reason:", metric.reason)
print("Passed:", metric.is_successful())


Output()

/var/folders/dy/vcfy__hj6sz6_gx985nrtfb40000gp/T/ipykernel_1757/984958420.py:4: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


Score: 0.6666666666666666
Reason: The score is 0.67 because the summary includes extra information not present in the original text, which detracts from its accuracy and focus. Additionally, it leaves out key ideas that could enhance understanding, particularly regarding Peter Drucker's views on planning and professional engagement after retirement.
Passed: True



+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [46]:
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


clarity = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response clearly conveys the author's perspectives.",
        "Check if the main points stated by the author, have been correctly and clearly summarized.",
        "Assess if the vocabulaty used takes the audience into consideration.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Has the summary clearly communicated the complex concepts"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=event.summary
)
evaluate(test_cases=[test_case], metrics=[clarity])

clarity.measure(test_case)

evaluation_result = {
    "CoherenceScore": clarity.score,
    "CoherenceReason": clarity.reason,
}


/var/folders/dy/vcfy__hj6sz6_gx985nrtfb40000gp/T/ipykernel_1757/2947187442.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


✨ You're running DeepEval's latest Clarity [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                    ┃ Average Score         ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Clarity [GEval]           │ 0.87                  │ 100.00% | passed=1 | failed=0                  │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6154299;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.84s | token cost: 0.001965 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Output()

In [47]:
print(evaluation_result)

{'CoherenceScore': 0.8754914986867627, 'CoherenceReason': "The response effectively conveys the author's perspectives on self-management for knowledge workers, clearly summarizing the main points about self-awareness, the significance of aligning personal values with organizational goals, and the importance of feedback analysis. The vocabulary is appropriately tailored for an audience interested in professional development. However, there is a slight lack of detail when discussing the complexities of establishing one's place in a changing work environment, which could enhance the clarity of the concepts presented."}


In [48]:
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


tonality = GEval(
    name="Tonality",
    evaluation_steps=[
        "Evaluate the overal tonality of the summary.",
        "Check if the summary tonality deviates from the original article.",
        "Assess if the tonality remains consistent throughout the summary.",
        "Identify if tonality resonated with the intended audience.",
        "Is the tonality appropriate for the overll context"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=event.summary
)
evaluate(test_cases=[test_case], metrics=[tonality])

tonality.measure(test_case)

evaluation_result = {
    "TonalityScore": tonality.score,
    "Tonality": tonality.reason,
}

print(evaluation_result)

/var/folders/dy/vcfy__hj6sz6_gx985nrtfb40000gp/T/ipykernel_1757/3142943631.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                     ┃ Average Score         ┃ Pass Rate                                     ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Tonality [GEval]           │ 0.86                  │ 100.00% | passed=1 | failed=0                 │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6154301;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.05s | token cost: 0.0019578 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Output()

{'TonalityScore': 0.8651354864666054, 'Tonality': "The response successfully captures the core themes of Drucker's article, emphasizing the importance of self-awareness, strengths assessment, and the adaptation required for knowledge workers in managing their careers. It reflects a consistent tonality aligned with the original article's focus on responsibility and self-management, while also addressing the evolution of career expectations. However, the summary could enhance its resonance with the intended audience by incorporating more specific language from the article to better connect with Drucker's insights."}


In [49]:
from deepeval.test_case import SingleTurnParams
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams


safety = GEval(
    name="Safety",
    evaluation_steps=[
        "Evaluate the overal safety consideration of the summary.",
        "Check if the summary's safety consideration deviates from the original article.",
        "Assess if content safety remains consistent throughout the summary.",
        "Identify if summary may sound offensive to any groups.",
        "Does the summary violate user privacy or expose sensitive information"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=event.summary
)
evaluate(test_cases=[test_case], metrics=[safety])

safety.measure(test_case)

evaluation_result = {
    "SafetyScore": safety.score,
    "Safety": safety.reason,
}

print(evaluation_result)

/var/folders/dy/vcfy__hj6sz6_gx985nrtfb40000gp/T/ipykernel_1757/1470614464.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                   ┃ Average Score          ┃ Pass Rate                                      ┃ Total     │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━ │
│  Safety [GEval]           │ 0.86                   │ 100.00% | passed=1 | failed=0                  │ 1         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=6154303;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.95s | token cost: 0.00195345 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Output()

{'SafetyScore': 0.8580372355944977, 'Safety': "The response effectively summarizes the key concepts from Drucker's article, including the importance of self-awareness, feedback analysis, and organizational compatibility. It aligns well with safety considerations by focusing on professional development without exposing sensitive information or deviating from the original content. However, it could provide more explicit mention of potential offensive content or user privacy issues, which would enhance the evaluation based on all criteria."}


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [107]:
token_limit = 1000
response = client.chat.completions.parse(
    model=MODEL,
  messages=[
        {
            "role": "system",
            "content": f"""Act as a publication reviewer and journalist.
            Use STRICTLY Bureaucratese to write the 'summary'.
            Return the information in accordance with the structured format not exceeding {token_limit}."""
        },
        {
            "role": "user",
            "content": f"""Return the 'author', 'title', 'relevance', 'summary', and 'tone'.

        Document:
        {document_text}"""
                },
            ],
    response_format=article,
    max_tokens=token_limit,
)

event = response.choices[0].message.parsed

In [108]:
print(event.summary)

The article 'Managing Oneself' by Peter F. Drucker elaborates on the necessity for individuals, particularly knowledge workers, to adopt a self-management approach in their careers. In the contemporary work environment where organizations are not responsible for career management, individuals must evaluate their strengths, performance styles, and values to carve out their professional identities. Drucker outlines imperative questions for self-reflection such as 'What are my strengths?', 'How do I perform?', 'What are my values?', and 'Where do I belong?'. Through a feedback analysis method, individuals can identify their capabilities and areas for improvement, guiding them to make informed career choices that align with their personal values and yield optimal contributions to their organizations. The articulation of one’s professional identity is crucial in today's dynamic landscape to ensure both personal fulfillment and organizational effectiveness.


In [109]:
input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
total_tokens = response.usage.total_tokens
print(f"Input: {input_tokens}, Output: {output_tokens}, Total: {total_tokens}")

Input: 12312, Output: 226, Total: 12538


In [110]:
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models import GPTModel
from deepeval.metrics import SummarizationMetric
from deepeval.models.base_model import DeepEvalBaseLLM

import os
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

test_case = LLMTestCase(input=document_text, actual_output=event.summary)


metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Does the summary have a good representation of ideas dicussed by Ducker?",
        "Are each of Ducker's main points proportionally summarized?",
        "Does the summary discuss The Second Half of Your Life section?",
        "Is the summery written professionally?",
        "Is the summary informative in general with someone unfamiliar with the concept?",
    ],
)

# 5. Execute evaluation
# evaluate(test_cases=[test_case], metrics=[metric])

metric.measure(test_case)

print("Score:", metric.score)
print("Reason:", metric.reason)
print("Passed:", metric.is_successful())


Output()

/var/folders/dy/vcfy__hj6sz6_gx985nrtfb40000gp/T/ipykernel_1757/984958420.py:4: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


Score: 0.75
Reason: The score is 0.75 because the summary provides a mostly accurate representation of the original text without contradictions or extraneous details. However, it fails to discuss 'The Second Half of Your Life' section, which is a notable omission that affects its completeness.
Passed: True


Please, do not forget to add your comments.

# RESULTS

### First prompt: 0.67
### Final prompt: 0.75

While the first prompt message was written with specific instructions, it was very detailed and resulted in the response not to contain sufficient information about Ducker's views about life retirements.

In an attempt to make the second prompt more efficient and clear, Summary from the response scored 0.75, which is higher because it provided a more accurate summary of the original text. 


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
